# 9.8 束搜索

解码时，贪心搜索每一步只保留当前概率最高的词元；束搜索会同时保留若干个候选序列，在质量和计算量之间折中。


In [ ]:
import math
import torch


## 贪心搜索示意


In [ ]:
def greedy_decode_step(log_probs):
    """返回每一步概率最大的词元索引。"""
    return int(torch.argmax(log_probs))

log_probs = torch.log(torch.tensor([0.1, 0.6, 0.3]))
print(greedy_decode_step(log_probs))


## 简单束搜索示意

下面的函数只展示束搜索的核心思想：每一步扩展候选序列，再保留得分最高的 `beam_size` 个。


In [ ]:
def simple_beam_search(step_log_probs, beam_size, eos_id=None):
    beams = [([], 0.0)]
    for log_probs in step_log_probs:
        candidates = []
        for seq, score in beams:
            if eos_id is not None and len(seq) > 0 and seq[-1] == eos_id:
                candidates.append((seq, score))
                continue
            for token_id, token_log_prob in enumerate(log_probs):
                candidates.append((seq + [token_id], score + float(token_log_prob)))
        beams = sorted(candidates, key=lambda x: x[1] / (len(x[0]) ** 0.75), reverse=True)[:beam_size]
    return beams

steps = [torch.log(torch.tensor([0.5, 0.4, 0.1])),
         torch.log(torch.tensor([0.2, 0.2, 0.6])),
         torch.log(torch.tensor([0.3, 0.5, 0.2]))]
print(simple_beam_search(steps, beam_size=2))


束大小越大，搜索范围越广，但计算和内存开销也越大。机器翻译中常用长度惩罚避免模型偏向过短句子。
